In [1]:
!pip install -q -U transformers torch accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 67.9 MB/s eta 0:00:00


In [3]:
import pandas as pd
import torch
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [5]:
from google.colab import files
uploaded = files.upload()

Saving DEV_TEST_Merged_ASQE_N2500.jsonl to DEV_TEST_Merged_ASQE_N2500 (1).jsonl
Saving Train_validation_Merged_ASQE_N4000.jsonl to Train_validation_Merged_ASQE_N4000 (1).jsonl


In [9]:
def load_jsonl(file_path):
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

train_df = load_jsonl("Train_validation_Merged_ASQE_N4000.jsonl")
test_df = load_jsonl("DEV_TEST_Merged_ASQE_N2500.jsonl")

print(train_df.columns)
train_df.head()

Index(['text', 'labels'], dtype='object')


,text,labels
0,He is not helpful at all and makes fun of stud...,"[{'aspect': 'He', 'opinion': 'Very rude', 'pol..."
1,"This course was amazing. For the first time, t...","[{'aspect': 'This course', 'opinion': 'I loved..."
2,Exams were very easy and he allowed us to do e...,"[{'aspect': 'Exams', 'opinion': 'very easy', '..."
3,The lecturers from Exeter have helped me throu...,"[{'aspect': 'lecturers', 'opinion': 'helped me..."
4,"Extremely boring and lectures were useless, bu...","[{'aspect': 'lectures', 'opinion': 'Extremely ..."


In [10]:
def convert_to_absa(df):
    rows = []

    for _, row in df.iterrows():
        sentence = row["text"]
        label_list = row["labels"]

        for item in label_list:
            aspect = item.get("aspect")
            sentiment = item.get("polarity")

            if aspect is None or aspect == "null":
                continue

            rows.append({
                "sentence": sentence,
                "aspect": aspect,
                "sentiment": sentiment
            })

    df_new = pd.DataFrame(rows)
    df_new = df_new.drop_duplicates().reset_index(drop=True)

    return df_new

train_absa = convert_to_absa(train_df)
test_absa = convert_to_absa(test_df)

train_absa["sentiment"] = train_absa["sentiment"].str.capitalize()
test_absa["sentiment"] = test_absa["sentiment"].str.capitalize()

print("Train size:", train_absa.shape)
print("Test size:", test_absa.shape)

test_absa.head()

Train size: (10331, 3)
Test size: (6600, 3)


,sentence,aspect,sentiment
0,She will only accept. docx forms in microsoft ...,This professor,Negative
1,Great Teacher! Learned a lot! If coming to cla...,Teacher,Positive
2,Not so easy class; great instructor!,class,Negative
3,Not so easy class; great instructor!,instructor,Positive
4,Studied for the exam literally the day of... e...,textbook,Negative


In [11]:
model_name = "answerdotai/ModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

model.to(device)

labels = ["Negative", "Neutral", "Positive"]

print("ModernBERT loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ModernBERT loaded successfully


In [13]:
batch_size = 32  # increase to 64 if GPU allows

predictions = []

for i in range(0, len(test_absa), batch_size):
    batch = test_absa.iloc[i:i+batch_size]

    texts = [
        f"{row['sentence']} [SEP] aspect: {row['aspect']}"
        for _, row in batch.iterrows()
    ]

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1).cpu().numpy()

    predictions.extend([labels[p] for p in preds])

    if i % 500 == 0:
        print(f"Processed {i} / {len(test_absa)}")

test_absa["modernbert_pred"] = predictions

Processed 0 / 6600
Processed 4000 / 6600


In [14]:
accuracy = accuracy_score(
    test_absa["sentiment"],
    test_absa["modernbert_pred"]
)

f1 = f1_score(
    test_absa["sentiment"],
    test_absa["modernbert_pred"],
    average="weighted"
)

print("ModernBERT Accuracy:", accuracy)
print("ModernBERT F1:", f1)

ModernBERT Accuracy: 0.4656060606060606
ModernBERT F1: 0.36236257964446983


In [15]:
print(classification_report(
    test_absa["sentiment"],
    test_absa["modernbert_pred"]
))

              precision    recall  f1-score   support

    Negative       0.31      0.02      0.03      2246
     Neutral       0.10      0.09      0.10       845
    Positive       0.51      0.84      0.64      3509

    accuracy                           0.47      6600
   macro avg       0.31      0.32      0.26      6600
weighted avg       0.39      0.47      0.36      6600

